# Imports

In [5]:
import json
import requests
import pandas as pd
from datetime import datetime, timezone  

# Global Variables

In [4]:
STATUS_URL ="https://toronto.publicbikesystem.net/customer/gbfs/v2/en/station_status"

# Fetch Station Status Data

In [8]:
response = requests.get(STATUS_URL, timeout=10)
response.status_code

200

# Explore Data Structure

In [14]:
data = response.json()
type(data)

dict

In [17]:
data.keys()

dict_keys(['last_updated', 'ttl', 'data', 'version'])

In [25]:
print(data['ttl'])
print(data['version'])
print(data['last_updated'])

6
2.3
1791822491


In [23]:
datetime.fromtimestamp(data['last_updated'],tz = timezone.utc)

datetime.datetime(2026, 10, 12, 16, 28, 11, tzinfo=datetime.timezone.utc)

In [26]:
data["data"].keys()

dict_keys(['stations'])

In [28]:
stations = data["data"]["stations"]

In [30]:
print(type(stations))
print(len(stations))

<class 'list'>
1075


In [43]:
len(stations[0].keys())

14

In [45]:
print(json.dumps(stations[0], indent=2))

{
  "station_id": "7000",
  "num_bikes_available": 40,
  "num_bikes_disabled": 0,
  "is_charging_station": false,
  "status": "IN_SERVICE",
  "traffic": null,
  "num_docks_available": 7,
  "num_docks_disabled": 0,
  "last_reported": 1791822457,
  "is_installed": true,
  "is_renting": true,
  "is_returning": true,
  "vehicle_docks_available": [
    {
      "vehicle_type_ids": [
        "ICONIC",
        "BOOST",
        "EFIT",
        "ASTRO"
      ],
      "count": 7
    }
  ],
  "vehicle_types_available": [
    {
      "vehicle_type_id": "BOOST",
      "count": 0
    },
    {
      "vehicle_type_id": "ICONIC",
      "count": 40
    },
    {
      "vehicle_type_id": "ASTRO",
      "count": 0
    },
    {
      "vehicle_type_id": "EFIT",
      "count": 0
    }
  ]
}


In [51]:
data["last_updated"] - stations[0]["last_reported"]   # → 34 (seconds)


34

In [54]:
from collections import Counter

key_counts = Counter()           # a dict that counts things, starting at 0
for station in stations:         # go through each of the 1,075 stations
    key_counts.update(station.keys())   # add 1 for each key this station has

key_counts


Counter({'station_id': 1075,
         'num_bikes_available': 1075,
         'num_bikes_disabled': 1075,
         'is_charging_station': 1075,
         'status': 1075,
         'traffic': 1075,
         'num_docks_available': 1075,
         'num_docks_disabled': 1075,
         'last_reported': 1075,
         'is_installed': 1075,
         'is_renting': 1075,
         'is_returning': 1075,
         'vehicle_docks_available': 1075,
         'vehicle_types_available': 1075})

# Convert to Dataframe

In [57]:
df = pd.json_normalize(stations)

In [58]:
df.head()

,station_id,num_bikes_available,num_bikes_disabled,is_charging_station,status,traffic,num_docks_available,num_docks_disabled,last_reported,is_installed,is_renting,is_returning,vehicle_docks_available,vehicle_types_available
0,7000,40,0,False,IN_SERVICE,None,7,0,1791822457,True,True,True,"[{'vehicle_type_ids': ['ICONIC', 'BOOST', 'EFI...","[{'vehicle_type_id': 'BOOST', 'count': 0}, {'v..."
1,7001,12,9,True,IN_SERVICE,None,2,0,1791822394,True,True,True,"[{'vehicle_type_ids': ['ICONIC', 'BOOST', 'EFI...","[{'vehicle_type_id': 'BOOST', 'count': 0}, {'v..."
2,7002,2,1,False,IN_SERVICE,None,16,0,1791822404,True,True,True,"[{'vehicle_type_ids': ['ICONIC', 'BOOST', 'EFI...","[{'vehicle_type_id': 'BOOST', 'count': 0}, {'v..."
3,7003,1,1,False,IN_SERVICE,None,13,0,1791822435,True,True,True,"[{'vehicle_type_ids': ['ICONIC', 'BOOST', 'EFI...","[{'vehicle_type_id': 'BOOST', 'count': 0}, {'v..."
4,7004,9,2,False,IN_SERVICE,None,24,0,1791822416,True,True,True,"[{'vehicle_type_ids': ['ICONIC', 'BOOST', 'EFI...","[{'vehicle_type_id': 'BOOST', 'count': 0}, {'v..."


# Analyze dataframe

In [59]:
df.dtypes

station_id                    str
num_bikes_available         int64
num_bikes_disabled          int64
is_charging_station          bool
status                        str
traffic                    object
num_docks_available         int64
num_docks_disabled          int64
last_reported               int64
is_installed                 bool
is_renting                   bool
is_returning                 bool
vehicle_docks_available    object
vehicle_types_available    object
dtype: object

In [62]:
df["status"].value_counts(dropna=False)

status
IN_SERVICE    1075
Name: count, dtype: int64

In [75]:
df["traffic"].value_counts(dropna=False)

traffic
None    1075
Name: count, dtype: int64

In [76]:
df["is_charging_station"].value_counts(dropna=False)

is_charging_station
False    1023
True       52
Name: count, dtype: int64

In [77]:
def docks_total(lst):
    return sum(item["count"] for item in lst)

(df["vehicle_docks_available"].apply(docks_total) == df["num_docks_available"]).all()


np.True_

In [78]:
def types_to_dict(lst):
    return {item["vehicle_type_id"]: item["count"] for item in lst}

types_df = df["vehicle_types_available"].apply(types_to_dict).apply(pd.Series)
types_df.head()


,BOOST,ICONIC,ASTRO,EFIT
0,0,40,0,0
1,0,1,0,11
2,0,2,0,0
3,0,1,0,0
4,0,8,0,1


In [79]:
(types_df.sum(axis=1) == df["num_bikes_available"]).all()


np.True_

In [80]:
df["vehicle_docks_available"].apply(len).value_counts()


vehicle_docks_available
1    1075
Name: count, dtype: int64

In [81]:
df["vehicle_docks_available"].apply(lambda lst: tuple(sorted(lst[0]["vehicle_type_ids"]))).value_counts()


vehicle_docks_available
(ASTRO, BOOST, EFIT, ICONIC)    1075
Name: count, dtype: int64

# Fetch Vehicle Type

In [82]:
def fetch_feed(url):
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return response.json()


In [ ]:
VEHICLE_TYPES_URL = "https://toronto.publicbikesystem.net/customer/gbfs/v2/en/vehicle_types"

vt = fetch_feed(VEHICLE_TYPES_URL)
vt["data"].keys()

dict_keys(['vehicle_types'])

In [84]:
vt_df = pd.json_normalize(vt["data"]["vehicle_types"])
vt_df


,vehicle_type_id,form_factor,propulsion_type,max_range_meters,name,default_pricing_plan_id,pricing_plan_ids
0,ICONIC,bicycle,human,0.0,ICONIC,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
1,FIT,bicycle,human,0.0,FIT,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
2,BOOST,bicycle,electric_assist,0.0,BOOST,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
3,EFIT,bicycle,electric_assist,60000.0,EFIT,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
4,EFIT G5,bicycle,electric_assist,60000.0,EFIT G5,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
5,CHLOE,scooter_standing,electric,0.0,CHLOE,274-120,"[274-120, 274-119, 275-124, 275-119, 276-120, ..."
6,COSMO,bicycle,electric_assist,60000.0,COSMO,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
7,ASTRO,bicycle,electric_assist,70000.0,ASTRO,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
8,METRO_BASE,bicycle,human,0.0,METRO_BASE,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."
9,METRO_CONNECTED,bicycle,human,0.0,METRO_CONNECTED,186-120,"[186-120, 186-119, 208-124, 208-119, 209-120, ..."


# Fetch Station Information

In [85]:
INFO_URL = "https://toronto.publicbikesystem.net/customer/gbfs/v2/en/station_information"

info = fetch_feed(INFO_URL)
info["data"].keys()

dict_keys(['stations'])

In [86]:
info_df = pd.json_normalize(info["data"]["stations"])
print(len(info_df))
info_df.head()


1075


,station_id,external_id,name,physical_configuration,lat,lon,altitude,address,capacity,is_charging_station,...,groups,obcn,short_name,nearby_distance,_bluetooth_id,_ride_code_support,post_code,is_valet_station,_valet_station_details,cross_street
0,7000,f8aa7900-e9c7-11ef-984e-124c66246c7d,Fort York Blvd / Capreol Ct,REGULAR,43.639707,-79.396143,0.0,Fort York Blvd / Capreol Ct,47,False,...,"[South, Fort York - Entertainment District]",647-643-9607,647-643-9607,500.0,,True,NaN,NaN,NaN,NaN
1,7001,f8aaeb6e-e9c7-11ef-984e-124c66246c7d,Wellesley Station Green P,ELECTRICBIKESTATION,43.664964,-79.383550,NaN,Yonge / Wellesley,23,True,...,"[E-Charging , South, Church Wellesley / Yorkvi...",416-617-9576,416-617-9576,500.0,,True,M4Y 1G7,NaN,NaN,NaN
2,7002,f8aaef03-e9c7-11ef-984e-124c66246c7d,St. George St / Bloor St W,REGULAR,43.667131,-79.399555,NaN,St. George St / Bloor St W,19,False,...,"[South, U of T - Hospital Row]",647-643-9615,647-643-9615,500.0,,True,NaN,NaN,NaN,NaN
3,7003,f8ab20ac-e9c7-11ef-984e-124c66246c7d,Madison Ave / Bloor St W,REGULAR,43.667018,-79.402796,NaN,Madison Ave / Bloor St W,15,False,...,"[South, Bloor St W / Annex]",647-631-4587,647-631-4587,500.0,,True,NaN,NaN,NaN,NaN
4,7004,f8ab24fe-e9c7-11ef-984e-124c66246c7d,University Ave / Elm St,VAULT,43.656499,-79.389362,0.0,University Ave / Elm St,35,False,...,[P7004-7047],647-643-9673,647-643-9673,500.0,,True,NaN,NaN,NaN,NaN


In [87]:
info_df.dtypes


station_id                    str
external_id                   str
name                          str
physical_configuration        str
lat                       float64
lon                       float64
altitude                  float64
address                       str
capacity                    int64
is_charging_station          bool
geofenced_capacity          int64
rental_methods             object
is_virtual_station           bool
groups                     object
obcn                          str
short_name                    str
nearby_distance           float64
_bluetooth_id                 str
_ride_code_support           bool
post_code                     str
is_valet_station           object
_valet_station_details        str
cross_street                  str
dtype: object

In [88]:
info_df.loc[info_df["station_id"] == "7000", ["name", "capacity"]]

,name,capacity
0,Fort York Blvd / Capreol Ct,47
